# Phase 1A Source Registry Review

Phase 1A builds or refreshes the AMC source registry. It does **not** download portfolio disclosure files, parse documents, or load PostgreSQL.

Source rule:

- **AMFI and SEBI** are reference/source-discovery layers.
- **AMC/provider websites** are the primary ingestion sources.

This notebook prefers existing artifacts and is non-mutating by default.

## 1. Imports and paths

In [32]:
from pathlib import Path
import json
import subprocess
import sys
from urllib.parse import urlparse

import pandas as pd
from IPython.display import Image, display


def find_repo_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'mutual_fund_ingestion' / '__init__.py').is_file():
            return candidate
    raise RuntimeError('Run this notebook from inside the Financial Analytics Work repository.')


def available_columns(frame, requested):
    return [column for column in requested if column in frame.columns]


def show_columns(frame, requested, *, empty_message='No matching records.'):
    columns = available_columns(frame, requested)
    if frame.empty:
        print(empty_message)
        return frame.reindex(columns=columns)
    return frame.loc[:, columns]


def read_jsonl(path):
    if not path.exists():
        print(f'Missing optional artifact: {path}')
        return []
    records = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), 1):
        if line.strip():
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                print(f'Skipping invalid JSONL record {path}:{line_number}: {exc}')
    return records

ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ROOT

from mutual_fund_ingestion.registry import load_registry
from mutual_fund_ingestion.source_registry import SourceRegistryPaths, calculate_source_registry_metrics

REGISTRY_PATH = ROOT / 'configs/amc_sources.yaml'
SOURCE_PATHS = SourceRegistryPaths.from_roots(
    REGISTRY_PATH,
    ROOT / 'data/raw/mutual_funds/source_registry',
    ROOT / 'data/reports/mutual_funds',
)
PATHS = {
    'registry_config': REGISTRY_PATH,
    'candidate_history': SOURCE_PATHS.candidates,
    'latest_snapshot': SOURCE_PATHS.latest,
    'source_registry_report': SOURCE_PATHS.report_html,
}
pd.DataFrame([{'artifact': name, 'path': str(path), 'exists': path.exists()} for name, path in PATHS.items()])

,artifact,path,exists
0,registry_config,/Users/vedaangchopra/all_data/complete_technic...,True
1,candidate_history,/Users/vedaangchopra/all_data/complete_technic...,True
2,latest_snapshot,/Users/vedaangchopra/all_data/complete_technic...,True
3,source_registry_report,/Users/vedaangchopra/all_data/complete_technic...,False


## 2. Load `configs/amc_sources.yaml`

In [33]:
if REGISTRY_PATH.exists():
    registry_entries = load_registry(REGISTRY_PATH)
    registry_df = pd.DataFrame([entry.to_dict() for entry in registry_entries])
else:
    print(f'Missing required registry: {REGISTRY_PATH}')
    registry_entries = []
    registry_df = pd.DataFrame()

show_columns(registry_df, [
    'amc_name', 'source_name', 'seed_url', 'amc_website', 'enabled', 'source_role',
    'source_type', 'priority', 'expected_document_types', 'discovered_from', 'notes'
])

,amc_name,source_name,seed_url,amc_website,enabled,source_role,source_type,priority,expected_document_types,discovered_from,notes
0,360 ONE Mutual Fund,NaN,https://www.360.one/asset/mutual-funds/downloads/,None,True,primary_provider,provider_download_page,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",
1,Abakkus Mutual Fund,NaN,https://www.abakkusmf.com/statutory-disclosure...,None,True,primary_provider,provider_homepage,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",Homepage fallback; refine from profile evidence.
2,Aditya Birla Sun Life Mutual Fund,NaN,https://mutualfund.adityabirlacapital.com/form...,None,True,primary_provider,provider_disclosure_page,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",
3,AlphaGrep Mutual Fund,NaN,https://alphagrepmf.com/,None,True,primary_provider,provider_homepage,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",Homepage fallback; refine from profile evidence.
4,Angel One Mutual Fund,NaN,https://www.angelonemf.com/,None,True,primary_provider,provider_homepage,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",Homepage fallback; refine from profile evidence.
5,ASK Mutual Fund,NaN,https://www.askfinancials.com/,None,True,primary_provider,provider_homepage,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",Homepage fallback; refine from profile evidence.
6,Axis Mutual Fund,NaN,https://www.axismf.com/statutory-disclosures,None,True,primary_provider,provider_disclosure_page,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",
7,Bajaj Finserv Mutual Fund,NaN,https://www.bajajamc.com/downloads?statutory-d...,None,True,primary_provider,provider_download_page,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",
8,Bandhan Mutual Fund,NaN,https://bandhanmutual.com/downloads/disclosures,None,True,primary_provider,provider_download_page,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",
9,Bank of India Mutual Fund,NaN,https://www.boimf.in/regulatory-reports,None,True,primary_provider,provider_download_page,primary,"(factsheet, portfolio_disclosure)","(manual_curated, existing_config)",


## 3. Load Phase 1A artifacts

In [34]:
candidate_records = read_jsonl(SOURCE_PATHS.candidates)
candidate_df = pd.DataFrame(candidate_records)

if SOURCE_PATHS.latest.exists():
    latest_records = json.loads(SOURCE_PATHS.latest.read_text(encoding='utf-8'))
else:
    print(f'Missing optional artifact: {SOURCE_PATHS.latest}')
    latest_records = []
latest_df = pd.DataFrame(latest_records)

print(f'Candidate records: {len(candidate_df)}')
print(f'Latest registry records: {len(latest_df)}')
display(show_columns(candidate_df, ['amc_name', 'source_name', 'seed_url', 'source_role', 'source_type', 'discovered_from', 'confidence']))
display(show_columns(latest_df, ['amc_name', 'source_name', 'seed_url', 'source_role', 'source_type', 'discovered_from', 'unresolved_reasons']))

Candidate records: 110
Latest registry records: 55


,amc_name,source_name,seed_url,source_role,source_type,discovered_from,confidence
0,360 ONE Mutual Fund,NaN,https://www.360one.com/asset-management/mutual...,primary_provider,provider_download_page,manual_curated,unknown
1,360 ONE Mutual Fund,NaN,https://www.360one.com/asset-management/mutual...,primary_provider,provider_download_page,existing_config,unknown
2,Abakkus Mutual Fund,NaN,https://www.abakkusmf.com/statutory-disclosure...,primary_provider,provider_homepage,manual_curated,unknown
3,Abakkus Mutual Fund,NaN,https://www.abakkusmf.com/statutory-disclosure...,primary_provider,provider_homepage,existing_config,unknown
4,Aditya Birla Sun Life Mutual Fund,NaN,https://mutualfund.adityabirlacapital.com/form...,primary_provider,provider_disclosure_page,manual_curated,unknown
...,...,...,...,...,...,...,...
105,Zerodha Mutual Fund,NaN,https://www.zerodhafundhouse.com/resources/,primary_provider,provider_download_page,existing_config,unknown
106,NaN,AMFI,https://www.amfiindia.com/,reference_index,industry_reference_portal,manual_reference,high
107,NaN,SEBI,https://www.sebi.gov.in/,reference_index,regulatory_reference_portal,manual_reference,medium
108,NaN,AMFI,https://www.amfiindia.com/,reference_index,industry_reference_portal,manual_reference,high


,amc_name,source_name,seed_url,source_role,source_type,discovered_from,unresolved_reasons
0,360 ONE Mutual Fund,NaN,https://www.360one.com/asset-management/mutual...,primary_provider,provider_download_page,"[manual_curated, existing_config]",[]
1,Abakkus Mutual Fund,NaN,https://www.abakkusmf.com/statutory-disclosure...,primary_provider,provider_homepage,"[manual_curated, existing_config]",[]
2,Aditya Birla Sun Life Mutual Fund,NaN,https://mutualfund.adityabirlacapital.com/form...,primary_provider,provider_disclosure_page,"[manual_curated, existing_config]",[]
3,AlphaGrep Mutual Fund,NaN,https://alphagrepmf.com/,primary_provider,provider_homepage,"[manual_curated, existing_config]",[]
4,Angel One Mutual Fund,NaN,https://www.angelonemf.com/,primary_provider,provider_homepage,"[manual_curated, existing_config]",[]
5,ASK Mutual Fund,NaN,https://www.askfinancials.com/,primary_provider,provider_homepage,"[manual_curated, existing_config]",[]
6,Axis Mutual Fund,NaN,https://www.axismf.com/statutory-disclosures,primary_provider,provider_disclosure_page,"[manual_curated, existing_config]",[]
7,Bajaj Finserv Mutual Fund,NaN,https://www.bajajamc.com/downloads,primary_provider,provider_download_page,"[manual_curated, existing_config]",[]
8,Bandhan Mutual Fund,NaN,https://bandhanmutual.com/downloads,primary_provider,provider_download_page,"[manual_curated, existing_config]",[]
9,Bank of India Mutual Fund,NaN,https://www.boimf.in/downloads,primary_provider,provider_download_page,"[manual_curated, existing_config]",[]


## 4. Optional small Phase 1A run

In [35]:
RUN_LIVE_SAMPLE = True
SAMPLE_LIMIT = 3

if RUN_LIVE_SAMPLE:
    command = [sys.executable, '-m', 'mutual_fund_ingestion', 'bootstrap-sources', '--dry-run', '--limit', str(SAMPLE_LIMIT)]
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True, check=False)
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
else:
    print('Live Phase 1A sample disabled. Set RUN_LIVE_SAMPLE = True to run bootstrap-sources --dry-run.')

{
  "metrics": {
    "total_sources": 55,
    "primary_provider_sources": 53,
    "reference_sources": 2,
    "amfi_reference_sources": 1,
    "sebi_reference_sources": 1,
    "manual_curated_sources": 53,
    "duplicate_merged_sources": 55,
    "sources_missing_seed_urls": 0,
    "sources_requiring_manual_completion": 0
  },
  "warnings": [
    "SEBI reference response is binary and unsupported for deterministic Phase 1A extraction."
  ],
  "sources": [
    {
      "enabled": true,
      "source_role": "primary_provider",
      "source_type": "provider_download_page",
      "seed_url": "https://www.360.one/asset/mutual-funds/downloads/",
      "amc_name": "360 ONE Mutual Fund",
      "source_name": null,
      "amc_website": null,
      "expected_document_types": [
        "factsheet",
        "portfolio_disclosure"
      ],
      "discovered_from": [
        "manual_curated",
        "existing_config"
      ],
      "confidence": "unknown",
      "priority": "primary",
      "manual_

## 5. Provenance and source summaries

In [36]:
summary_frames = {}
for column in ['discovered_from', 'source_role', 'source_type', 'priority', 'enabled']:
    if column not in registry_df.columns:
        continue
    if column == 'discovered_from':
        values = registry_df[column].explode()
    else:
        values = registry_df[column]
    summary_frames[column] = values.fillna('<missing>').value_counts(dropna=False).rename_axis(column).reset_index(name='count')
    display(summary_frames[column])

metrics = calculate_source_registry_metrics(registry_entries) if registry_entries else {}
pd.DataFrame([{'metric': key, 'value': value} for key, value in metrics.items()])

,discovered_from,count
0,manual_curated,53
1,existing_config,53
2,manual_reference,2


,source_role,count
0,primary_provider,53
1,reference_index,2


,source_type,count
0,provider_download_page,35
1,provider_homepage,12
2,provider_disclosure_page,6
3,industry_reference_portal,1
4,regulatory_reference_portal,1


,priority,count
0,primary,53
1,secondary,2


,enabled,count
0,True,55


,metric,value
0,total_sources,55
1,primary_provider_sources,53
2,reference_sources,2
3,amfi_reference_sources,1
4,sebi_reference_sources,1
5,manual_curated_sources,53
6,duplicate_merged_sources,0
7,sources_missing_seed_urls,0
8,sources_requiring_manual_completion,0


## 6. Registry quality checks

In [37]:
def normalized_domain(value):
    if not isinstance(value, str) or not value:
        return None
    return urlparse(value).netloc.casefold().removeprefix('www.') or None

quality = {}
if not registry_df.empty:
    quality['missing_amc_name'] = registry_df[registry_df.get('source_role', '').eq('primary_provider') & registry_df.get('amc_name', pd.Series(index=registry_df.index, dtype=object)).isna()]
    quality['missing_seed_url'] = registry_df[registry_df.get('seed_url', pd.Series(index=registry_df.index, dtype=object)).isna()]
    quality['missing_amc_website'] = registry_df[registry_df.get('source_role', '').eq('primary_provider') & registry_df.get('amc_website', pd.Series(index=registry_df.index, dtype=object)).isna()]
    if 'amc_name' in registry_df:
        names = registry_df['amc_name'].astype('string').str.casefold()
        quality['duplicate_amc_names'] = registry_df[names.notna() & names.duplicated(keep=False)]
    if 'seed_url' in registry_df:
        domains = registry_df['seed_url'].map(normalized_domain)
        quality['duplicate_domains'] = registry_df[domains.notna() & domains.duplicated(keep=False)]
    quality['disabled_sources'] = registry_df[~registry_df.get('enabled', True).fillna(False).astype(bool)]
    quality['unknown_source_role'] = registry_df[~registry_df.get('source_role', pd.Series(index=registry_df.index, dtype=object)).isin(['primary_provider', 'reference_index'])]
    if 'source_type' in registry_df:
        quality['unknown_source_type'] = registry_df[registry_df['source_type'].isna() | registry_df['source_type'].eq('unknown')]
    if 'expected_document_types' in registry_df:
        quality['empty_expected_document_types'] = registry_df[~registry_df['expected_document_types'].map(bool)]

quality_counts = pd.DataFrame([{'check': name, 'count': len(frame)} for name, frame in quality.items()])
display(quality_counts)
for name, frame in quality.items():
    if not frame.empty:
        print(f'--- {name} ({len(frame)}) ---')
        display(show_columns(frame, ['amc_name', 'source_name', 'seed_url', 'source_role', 'source_type', 'enabled', 'unresolved_reasons']))

,check,count
0,missing_amc_name,0
1,missing_seed_url,0
2,missing_amc_website,53
3,duplicate_amc_names,0
4,duplicate_domains,0
5,disabled_sources,0
6,unknown_source_role,0
7,unknown_source_type,0
8,empty_expected_document_types,0


--- missing_amc_website (53) ---


,amc_name,source_name,seed_url,source_role,source_type,enabled,unresolved_reasons
0,360 ONE Mutual Fund,NaN,https://www.360.one/asset/mutual-funds/downloads/,primary_provider,provider_download_page,True,()
1,Abakkus Mutual Fund,NaN,https://www.abakkusmf.com/statutory-disclosure...,primary_provider,provider_homepage,True,()
2,Aditya Birla Sun Life Mutual Fund,NaN,https://mutualfund.adityabirlacapital.com/form...,primary_provider,provider_disclosure_page,True,()
3,AlphaGrep Mutual Fund,NaN,https://alphagrepmf.com/,primary_provider,provider_homepage,True,()
4,Angel One Mutual Fund,NaN,https://www.angelonemf.com/,primary_provider,provider_homepage,True,()
5,ASK Mutual Fund,NaN,https://www.askfinancials.com/,primary_provider,provider_homepage,True,()
6,Axis Mutual Fund,NaN,https://www.axismf.com/statutory-disclosures,primary_provider,provider_disclosure_page,True,()
7,Bajaj Finserv Mutual Fund,NaN,https://www.bajajamc.com/downloads?statutory-d...,primary_provider,provider_download_page,True,()
8,Bandhan Mutual Fund,NaN,https://bandhanmutual.com/downloads/disclosures,primary_provider,provider_download_page,True,()
9,Bank of India Mutual Fund,NaN,https://www.boimf.in/regulatory-reports,primary_provider,provider_download_page,True,()


## 7. AMFI and reference-source inspection

In [38]:
if registry_df.empty:
    reference_df = registry_df
else:
    reference_mask = registry_df.get('source_role', '').eq('reference_index')
    reference_df = registry_df[reference_mask]

display(show_columns(reference_df, ['source_name', 'seed_url', 'source_role', 'source_type', 'priority', 'discovered_from', 'access_notes', 'notes']))

amfi_rows = reference_df[reference_df.get('source_name', pd.Series(index=reference_df.index, dtype=object)).astype('string').str.casefold().eq('amfi')] if not reference_df.empty else reference_df
amfi_valid = not amfi_rows.empty and amfi_rows.get('source_role', pd.Series(dtype=object)).eq('reference_index').all() and amfi_rows.get('priority', pd.Series(dtype=object)).eq('secondary').all()
print({'amfi_reference_configuration_valid': bool(amfi_valid), 'amfi_rows': len(amfi_rows)})
if not amfi_valid:
    print('WARNING: AMFI should be configured as source_role=reference_index and priority=secondary.')

,source_name,seed_url,source_role,source_type,priority,discovered_from,access_notes,notes
53,AMFI,https://www.amfiindia.com/,reference_index,industry_reference_portal,secondary,"(manual_reference,)",May require VPN depending on the execution env...,"Use for source discovery and validation, not a..."
54,SEBI,https://www.sebi.gov.in/,reference_index,regulatory_reference_portal,secondary,"(manual_reference,)",,Corroborative regulatory reference; discovered...


{'amfi_reference_configuration_valid': True, 'amfi_rows': 1}


## 8. Manual override inspection

In [39]:
if registry_df.empty or 'discovered_from' not in registry_df:
    manual_df = pd.DataFrame()
else:
    manual_mask = registry_df['discovered_from'].map(lambda values: bool(set(values or ()) & {'manual_curated', 'existing_config'}))
    manual_df = registry_df[manual_mask]
show_columns(manual_df, ['amc_name', 'seed_url', 'source_type', 'discovered_from', 'manual_overrides', 'notes'])

,amc_name,seed_url,source_type,discovered_from,manual_overrides,notes
0,360 ONE Mutual Fund,https://www.360.one/asset/mutual-funds/downloads/,provider_download_page,"(manual_curated, existing_config)","(seed_url, source_type)",
1,Abakkus Mutual Fund,https://www.abakkusmf.com/statutory-disclosure...,provider_homepage,"(manual_curated, existing_config)","(seed_url, source_type)",Homepage fallback; refine from profile evidence.
2,Aditya Birla Sun Life Mutual Fund,https://mutualfund.adityabirlacapital.com/form...,provider_disclosure_page,"(manual_curated, existing_config)","(seed_url, source_type)",
3,AlphaGrep Mutual Fund,https://alphagrepmf.com/,provider_homepage,"(manual_curated, existing_config)","(seed_url, source_type)",Homepage fallback; refine from profile evidence.
4,Angel One Mutual Fund,https://www.angelonemf.com/,provider_homepage,"(manual_curated, existing_config)","(seed_url, source_type)",Homepage fallback; refine from profile evidence.
5,ASK Mutual Fund,https://www.askfinancials.com/,provider_homepage,"(manual_curated, existing_config)","(seed_url, source_type)",Homepage fallback; refine from profile evidence.
6,Axis Mutual Fund,https://www.axismf.com/statutory-disclosures,provider_disclosure_page,"(manual_curated, existing_config)","(seed_url, source_type)",
7,Bajaj Finserv Mutual Fund,https://www.bajajamc.com/downloads?statutory-d...,provider_download_page,"(manual_curated, existing_config)","(seed_url, source_type)",
8,Bandhan Mutual Fund,https://bandhanmutual.com/downloads/disclosures,provider_download_page,"(manual_curated, existing_config)","(seed_url, source_type)",
9,Bank of India Mutual Fund,https://www.boimf.in/regulatory-reports,provider_download_page,"(manual_curated, existing_config)","(seed_url, source_type)",


## 9. Output artifacts

In [40]:
artifact_df = pd.DataFrame([{'artifact': name, 'path': str(path), 'exists': path.exists()} for name, path in PATHS.items()])
display(artifact_df)
if SOURCE_PATHS.report_html.exists():
    print(f'Source registry report: {SOURCE_PATHS.report_html}')

,artifact,path,exists
0,registry_config,/Users/vedaangchopra/all_data/complete_technic...,True
1,candidate_history,/Users/vedaangchopra/all_data/complete_technic...,True
2,latest_snapshot,/Users/vedaangchopra/all_data/complete_technic...,True
3,source_registry_report,/Users/vedaangchopra/all_data/complete_technic...,False


## 10. Readiness summary for Phase 1B

In [41]:
if registry_df.empty:
    readiness = {'total_registry_entries': 0, 'phase_1b_can_run': False, 'recommendation': 'Create or restore the source registry before Phase 1B.'}
else:
    enabled = registry_df.get('enabled', True).fillna(False).astype(bool)
    primary = registry_df.get('source_role', '').eq('primary_provider')
    has_seed = registry_df.get('seed_url', pd.Series(index=registry_df.index, dtype=object)).notna()
    unresolved = registry_df.get('unresolved_reasons', pd.Series([() for _ in registry_df.index], index=registry_df.index)).map(bool)
    enabled_primary = enabled & primary
    usable_primary = enabled_primary & has_seed
    missing_primary = enabled_primary & ~has_seed
    readiness = {
        'total_registry_entries': len(registry_df),
        'enabled_primary_provider_entries': int(enabled_primary.sum()),
        'usable_enabled_primary_entries': int(usable_primary.sum()),
        'reference_only_entries': int(registry_df.get('source_role', '').eq('reference_index').sum()),
        'entries_missing_seed_url': int(missing_primary.sum()),
        'entries_needing_manual_completion': int(unresolved.sum()),
        'phase_1b_can_run': bool(usable_primary.any()),
        'recommendation': 'Run bounded Phase 1B profiling.' if usable_primary.any() else 'Fix Phase 1A provider seed URLs before Phase 1B.',
    }
readiness

{'total_registry_entries': 55,
 'enabled_primary_provider_entries': 53,
 'usable_enabled_primary_entries': 53,
 'reference_only_entries': 2,
 'entries_missing_seed_url': 0,
 'entries_needing_manual_completion': 0,
 'phase_1b_can_run': True,
 'recommendation': 'Run bounded Phase 1B profiling.'}